# Gold Direction Prediction — UP or DOWN

This notebook predicts whether the next trading day's **closing price** will be higher or lower than the current day's close.

- **UP (1):** next day's close > today's close
- **DOWN (0):** next day's close <= today's close

This is a direction-classification task, not a volatility forecast. It uses daily COMEX Gold Futures data from Yahoo Finance (`GC=F`). The models only use information available at the end of the current trading day.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Load the gold data

The downloader from the project saves the daily data to `data/gold_daily.csv`. If the file does not exist, this cell downloads it.

In [ ]:
os.makedirs("data", exist_ok=True)
DATA_PATH = "data/gold_daily.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
else:
    df = yf.download(
        "GC=F",
        start="2000-01-01",
        interval="1d",
        auto_adjust=False,
        progress=False
    )
    if df.empty:
        raise RuntimeError("Yahoo Finance returned no data. Check your connection or ticker.")
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.to_csv(DATA_PATH)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.columns = [str(c).lower().replace(" ", "_") for c in df.columns]
df.index = pd.to_datetime(df.index)
df = df[~df.index.duplicated(keep="last")].sort_index()

required = ["open", "high", "low", "close", "volume"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[required].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=["open", "high", "low", "close"])
df.tail()

## 2. Create features

Each row describes the market using today's and previous days' information. The target is created separately and shifted from the next trading day.

In [ ]:
data = df.copy()

# Daily price changes and candle information
data["return_1d"] = data["close"].pct_change()
data["return_2d"] = data["close"].pct_change(2)
data["return_5d"] = data["close"].pct_change(5)
data["return_10d"] = data["close"].pct_change(10)
data["daily_range_pct"] = (data["high"] - data["low"]) / data["close"]
data["open_close_pct"] = (data["close"] - data["open"]) / data["open"]
data["high_close_pct"] = (data["high"] - data["close"]) / data["close"]
data["close_low_pct"] = (data["close"] - data["low"]) / data["close"]

# Moving averages and distance from them
for window in [5, 10, 20, 50, 100, 200]:
    ma = data["close"].rolling(window).mean()
    data[f"close_vs_ma_{window}"] = data["close"] / ma - 1

# Recent momentum and rolling volatility
for window in [5, 10, 20, 50]:
    data[f"momentum_{window}"] = data["close"].pct_change(window)
    data[f"return_std_{window}"] = data["return_1d"].rolling(window).std()

# Volume features
data["volume_change"] = data["volume"].pct_change()
data["volume_ma_ratio"] = data["volume"] / data["volume"].rolling(20).mean()

# Calendar features
data["day_of_week"] = data.index.dayofweek
data["day_sin"] = np.sin(2 * np.pi * data["day_of_week"] / 5)
data["day_cos"] = np.cos(2 * np.pi * data["day_of_week"] / 5)

# Target: 1 if next close is higher, otherwise 0
data["next_close"] = data["close"].shift(-1)
data = data.dropna(subset=["next_close"])  # Remove the final row: its next-day close is unknown\ndata["target"] = (data["next_close"] > data["close"]).astype(int)

feature_columns = [
    c for c in data.columns
    if c not in ["next_close", "target", "day_of_week"]
]

dataset = data[feature_columns + ["target"]].replace([np.inf, -np.inf], np.nan)
dataset = dataset.dropna()

X = dataset[feature_columns].astype(float)
y = dataset["target"].astype(int)

print("Rows:", len(dataset))
print("Features:", len(feature_columns))
print("\nTarget distribution:")
print(y.value_counts().rename(index={0: "DOWN", 1: "UP"}))
print("\nUP percentage:", round(y.mean() * 100, 2), "%")

## 3. Chronological train/test split

We do not shuffle financial time-series data. The first 80% is used for training and the last 20% for testing, so the test set represents later dates.

In [ ]:
split_index = int(len(dataset) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()
y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

test_dates = X_test.index

print("Training period:", X_train.index.min().date(), "to", X_train.index.max().date())
print("Testing period: ", X_test.index.min().date(), "to", X_test.index.max().date())
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 4. Evaluation function

Accuracy alone can be misleading if UP and DOWN do not occur equally often. We also report balanced accuracy, precision, recall and F1-score.

In [ ]:
def evaluate_model(name, y_true, predictions, probabilities=None):
    results = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Balanced accuracy": balanced_accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
    }
    if probabilities is not None and len(np.unique(y_true)) == 2:
        results["ROC AUC"] = roc_auc_score(y_true, probabilities)
    return results

results = []
predictions_by_model = {}

## 5. Logistic Regression

This is a simple baseline classifier. Scaling is fitted only on the training data through the pipeline.

In [ ]:
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

predictions_by_model["Logistic Regression"] = logistic_pred
results.append(evaluate_model("Logistic Regression", y_test, logistic_pred, logistic_prob))

## 6. Random Forest

Random Forest combines many decision trees. It can learn non-linear relationships between the indicators.

In [ ]:
forest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ))
])

forest_model.fit(X_train, y_train)
forest_pred = forest_model.predict(X_test)
forest_prob = forest_model.predict_proba(X_test)[:, 1]

predictions_by_model["Random Forest"] = forest_pred
results.append(evaluate_model("Random Forest", y_test, forest_pred, forest_prob))

## 7. PyTorch neural network

The network outputs a probability between 0 and 1 using a sigmoid. A probability above 0.5 means UP; otherwise it predicts DOWN.

In [ ]:
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_scaled = scaler.transform(imputer.transform(X_test))

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1, 1)

train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=128,
    shuffle=True
)

class DirectionNetwork(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

nn_model = DirectionNetwork(X_train_scaled.shape[1]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.001, weight_decay=1e-4)

EPOCHS = 30
for epoch in range(EPOCHS):
    nn_model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
        optimizer.zero_grad()
        logits = nn_model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(batch_X)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - loss: {epoch_loss / len(X_train_tensor):.4f}")

nn_model.eval()
with torch.no_grad():
    nn_logits = nn_model(X_test_tensor.to(DEVICE)).cpu().numpy().ravel()
    nn_prob = 1 / (1 + np.exp(-nn_logits))
    nn_pred = (nn_prob >= 0.5).astype(int)

predictions_by_model["PyTorch Neural Network"] = nn_pred
results.append(evaluate_model("PyTorch Neural Network", y_test, nn_pred, nn_prob))

## 8. Compare the models

In [ ]:
results_df = pd.DataFrame(results).set_index("Model").sort_values("F1", ascending=False)
display(results_df.round(4))

ax = results_df[["Accuracy", "Balanced accuracy", "F1"]].plot(
    kind="bar", figsize=(11, 5), ylim=(0, 1), rot=15
)
ax.set_title("Gold next-day direction — test performance")
ax.set_ylabel("Score")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Confusion matrix and classification report

The confusion matrix shows how many UP and DOWN days each model classified correctly and incorrectly.

In [ ]:
best_model_name = results_df.index[0]
best_predictions = predictions_by_model[best_model_name]

print("Best model by test F1:", best_model_name)
print(classification_report(
    y_test,
    best_predictions,
    labels=[0, 1],
    target_names=["DOWN", "UP"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_predictions,
    labels=[0, 1],
    display_labels=["DOWN", "UP"],
    cmap="Blues"
)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.show()

## 10. Inspect the predictions

This table compares the actual next-day direction with the model's prediction. For the neural network, the probability is its estimated probability of UP.

In [ ]:
prediction_output = pd.DataFrame(index=test_dates)
prediction_output["Actual"] = y_test.map({0: "DOWN", 1: "UP"})
prediction_output["Logistic prediction"] = pd.Series(logistic_pred, index=test_dates).map({0: "DOWN", 1: "UP"})
prediction_output["Random Forest prediction"] = pd.Series(forest_pred, index=test_dates).map({0: "DOWN", 1: "UP"})
prediction_output["Neural network prediction"] = pd.Series(nn_pred, index=test_dates).map({0: "DOWN", 1: "UP"})
prediction_output["Neural network UP probability"] = nn_prob

display(prediction_output.tail(20).round(3))

os.makedirs("outputs", exist_ok=True)
prediction_output.to_csv("outputs/gold_direction_test_predictions.csv")
results_df.to_csv("outputs/model_comparison.csv")
print("Saved predictions and model comparison in outputs/")

## Important notes

- This notebook predicts **UP or DOWN for the next trading day's close**, not the size of the move.
- The target uses the next day's close, but the input features stop at the current day. The chronological split prevents training on future rows when testing.
- A result close to 50% accuracy may mean the daily direction is difficult to predict from price history alone. Do not assume a model is useful just because it produces predictions.
- This is a research/learning project, not a trading system. Before considering real use, test transaction costs, slippage, out-of-sample periods and a simple baseline strategy.
- Yahoo Finance's `GC=F` is a gold futures ticker, not a continuous spot XAU/USD feed.